# Introduction to GPU Computing

Understand why parallel computing matters, how GPU architecture differs from CPUs, and when to use heterogeneous computing. Covers Moore's Law, the power wall, SIMT execution, and Amdahl's Law.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-intro-gpu)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Why Parallel Computing

In [ ]:
!pip install numba

import numpy as np
import time

# ============================================
# Demonstrating why parallel computing matters
# ============================================

# Task: Add two large vectors element-by-element
# This is "embarrassingly parallel" -- each element is independent

n = 10_000_000
a = np.random.randn(n).astype(np.float32)
b = np.random.randn(n).astype(np.float32)

# Method 1: Pure Python loop (sequential, one element at a time)
def python_add(a, b):
    c = np.empty_like(a)
    for i in range(len(a)):
        c[i] = a[i] + b[i]
    return c

# Time the pure Python approach (only first 100K to avoid waiting)
small_a, small_b = a[:100_000], b[:100_000]
start = time.perf_counter()
c_py = python_add(small_a, small_b)
python_time = time.perf_counter() - start
print(f"Pure Python (100K elements): {python_time*1000:.1f} ms")
print(f"  Estimated for 10M elements: {python_time*100*1000:.0f} ms")

# Method 2: NumPy vectorized (uses SIMD instructions on CPU)
start = time.perf_counter()
c_np = a + b
numpy_time = time.perf_counter() - start
print(f"\nNumPy vectorized (10M elements): {numpy_time*1000:.2f} ms")
print(f"  Speedup vs Python: {(python_time*100)/numpy_time:.0f}x")

# Method 3: GPU with numba.cuda
from numba import cuda

@cuda.jit
def gpu_add(a, b, c):
    idx = cuda.grid(1)  # Get global thread index
    if idx < a.size:     # Boundary check
        c[idx] = a[idx] + b[idx]

# Transfer data to GPU
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array_like(a)

# Configure grid: 256 threads per block
threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Warm up
gpu_add[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()

# Timed run
start = time.perf_counter()
gpu_add[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()
gpu_time = time.perf_counter() - start

c_gpu = d_c.copy_to_host()
print(f"\nGPU numba.cuda (10M elements): {gpu_time*1000:.2f} ms")
print(f"  Speedup vs NumPy: {numpy_time/gpu_time:.1f}x")
print(f"  Speedup vs Python: {(python_time*100)/gpu_time:.0f}x")

# Verify correctness
assert np.allclose(c_np, c_gpu, atol=1e-5), "Results don't match!"
print("\nResults verified: GPU output matches CPU output.")

### Lesson example: CPU vs GPU Architecture

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda

# ============================================
# Exploring GPU hardware properties
# ============================================

# Query GPU device information
device = cuda.get_current_device()
print("=== GPU Device Information ===")
print(f"Device name: {device.name}")
print(f"Compute capability: {device.compute_capability}")
print(f"Max threads per block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max block dimensions: {device.MAX_BLOCK_DIM_X} x {device.MAX_BLOCK_DIM_Y} x {device.MAX_BLOCK_DIM_Z}")
print(f"Max grid dimensions: {device.MAX_GRID_DIM_X} x {device.MAX_GRID_DIM_Y} x {device.MAX_GRID_DIM_Z}")
print(f"Warp size: {device.WARP_SIZE}")
print(f"Max shared memory per block: {device.MAX_SHARED_MEMORY_PER_BLOCK} bytes")
print(f"Multiprocessor count: {device.MULTIPROCESSOR_COUNT}")

# ============================================
# Demonstrating SIMT and warp divergence
# ============================================

# Kernel WITHOUT divergence: all threads take the same path
@cuda.jit
def no_divergence_kernel(data, out):
    idx = cuda.grid(1)
    if idx < data.size:
        # All threads do the same work
        out[idx] = data[idx] * 2.0 + 1.0

# Kernel WITH divergence: odd/even threads take different paths
@cuda.jit
def divergent_kernel(data, out):
    idx = cuda.grid(1)
    if idx < data.size:
        if idx % 2 == 0:  # Half the warp goes one way
            out[idx] = data[idx] * 2.0 + 1.0
        else:              # Other half goes the other way
            out[idx] = data[idx] * 3.0 - 1.0

n = 10_000_000
data = np.random.randn(n).astype(np.float32)
d_data = cuda.to_device(data)
d_out = cuda.device_array(n, dtype=np.float32)

threads = 256
blocks = (n + threads - 1) // threads

# Warm up
no_divergence_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
divergent_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()

# Benchmark: No divergence
iterations = 100
start = time.perf_counter()
for _ in range(iterations):
    no_divergence_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
no_div_time = (time.perf_counter() - start) / iterations

# Benchmark: With divergence
start = time.perf_counter()
for _ in range(iterations):
    divergent_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
div_time = (time.perf_counter() - start) / iterations

print(f"\n=== Warp Divergence Impact ===")
print(f"No divergence: {no_div_time*1000:.3f} ms")
print(f"With divergence: {div_time*1000:.3f} ms")
print(f"Slowdown from divergence: {div_time/no_div_time:.2f}x")
print("\nNote: Even simple odd/even branching causes measurable slowdown.")
print("In real kernels with complex branching, the penalty can be much worse.")

### Lesson example: Heterogeneous Computing

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda

# ============================================
# Heterogeneous Computing: Host/Device Model
# ============================================

# Step 1: Prepare data on HOST (CPU)
print("=== Host/Device Data Transfer Demo ===")
n = 5_000_000
host_a = np.random.randn(n).astype(np.float32)
host_b = np.random.randn(n).astype(np.float32)
print(f"Created {n:,} element arrays on CPU")
print(f"Data size: {host_a.nbytes / 1e6:.1f} MB per array")

# Step 2: Transfer to DEVICE (GPU) -- measure the cost
start = time.perf_counter()
d_a = cuda.to_device(host_a)
d_b = cuda.to_device(host_b)
d_c = cuda.device_array(n, dtype=np.float32)
cuda.synchronize()
transfer_to_time = time.perf_counter() - start
print(f"\nCPU -> GPU transfer: {transfer_to_time*1000:.2f} ms")
print(f"  Effective bandwidth: {3 * host_a.nbytes / transfer_to_time / 1e9:.1f} GB/s")

# Step 3: Compute on DEVICE
@cuda.jit
def vector_add(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]

threads = 256
blocks = (n + threads - 1) // threads

vector_add[blocks, threads](d_a, d_b, d_c)  # Warm up
cuda.synchronize()

start = time.perf_counter()
vector_add[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()
compute_time = time.perf_counter() - start
print(f"\nGPU compute: {compute_time*1000:.3f} ms")

# Step 4: Transfer results back to HOST
start = time.perf_counter()
host_c = d_c.copy_to_host()
transfer_from_time = time.perf_counter() - start
print(f"GPU -> CPU transfer: {transfer_from_time*1000:.2f} ms")

# Analysis: Where does the time go?
total = transfer_to_time + compute_time + transfer_from_time
print(f"\n=== Time Breakdown ===")
print(f"CPU -> GPU transfer: {transfer_to_time/total*100:.1f}%")
print(f"GPU computation:     {compute_time/total*100:.1f}%")
print(f"GPU -> CPU transfer: {transfer_from_time/total*100:.1f}%")
print(f"Total:               {total*1000:.2f} ms")

# ============================================
# Amdahl's Law Demonstration
# ============================================
print(f"\n=== Amdahl's Law ===")
print(f"Sequential fraction (transfers): {(transfer_to_time + transfer_from_time)/total*100:.1f}%")
print(f"Parallel fraction (compute):     {compute_time/total*100:.1f}%")
print(f"\nFor vector addition, transfer dominates because arithmetic intensity is LOW.")
print(f"(Only 1 FLOP per 12 bytes loaded -- 2 reads + 1 write of 4-byte floats)")

# ============================================
# Demonstrate: keeping data on GPU is key
# ============================================
print(f"\n=== Pipeline: Stay on GPU ===")

@cuda.jit
def scale_kernel(data, scalar):
    idx = cuda.grid(1)
    if idx < data.size:
        data[idx] *= scalar

# BAD: Transfer back after each step
start = time.perf_counter()
for _ in range(10):
    d_data = cuda.to_device(host_a)
    scale_kernel[blocks, threads](d_data, 1.01)
    cuda.synchronize()
    host_a_tmp = d_data.copy_to_host()
cuda.synchronize()
bad_time = time.perf_counter() - start

# GOOD: Stay on GPU for all 10 steps
d_data = cuda.to_device(host_a)
start = time.perf_counter()
for _ in range(10):
    scale_kernel[blocks, threads](d_data, 1.01)
cuda.synchronize()
result = d_data.copy_to_host()
good_time = time.perf_counter() - start

print(f"BAD  (transfer each step):  {bad_time*1000:.2f} ms")
print(f"GOOD (stay on GPU):         {good_time*1000:.2f} ms")
print(f"Speedup from avoiding transfers: {bad_time/good_time:.1f}x")

---

## Exercise: CPU vs GPU Vector Addition Benchmark


In [ ]:
import numpy as np
import time
from numba import cuda

# GPU kernel for vector addition
@cuda.jit
def gpu_vector_add(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]

def benchmark_size(n, num_iters=5):
    """Benchmark CPU vs GPU vector addition for a given size."""
    a = np.random.randn(n).astype(np.float32)
    b = np.random.randn(n).astype(np.float32)

    # CPU benchmark (NumPy)
    c_cpu = a + b  # Warm up
    start = time.perf_counter()
    for _ in range(num_iters):
        c_cpu = a + b
    cpu_time = (time.perf_counter() - start) / num_iters

    # GPU benchmark
    threads = 256
    blocks = (n + threads - 1) // threads

    # TODO: Measure GPU transfer time (host -> device)
    # TODO: Measure GPU compute time (kernel execution)
    # TODO: Measure GPU transfer back time (device -> host)
    # TODO: Calculate total GPU time
    # TODO: Verify correctness

    # Return: cpu_time, transfer_to, compute, transfer_from, total_gpu
    pass

# TODO: Test sizes [1_000, 100_000, 1_000_000, 10_000_000]
# TODO: Print formatted table with columns:
#   Size | CPU (ms) | GPU Total (ms) | GPU Compute (ms) | Transfer (ms) | Speedup | Ideal Speedup
# TODO: Identify the crossover point

## Your tasks

Implement vector addition on both CPU and GPU, measure transfer times, compute times, and total times, then analyze the results using Amdahl's Law.

**Requirements:**
- Create random float32 vectors of sizes: 1000, 100000, 1000000, 10000000
- For each size, measure: (a) CPU computation time (using NumPy), (b) GPU transfer time (to + from), (c) GPU computation time, (d) total GPU time
- Compute the speedup as CPU_time / GPU_total_time for each size
- Also compute the 'ideal' speedup as CPU_time / GPU_compute_time (ignoring transfers)
- Print results in a clear table
- Discuss: at what size does the GPU become worthwhile, considering transfer overhead?

**Hints:**
- Use `cuda.to_device()` and `.copy_to_host()` for transfers
- Use `cuda.synchronize()` before timing measurements
- Warm up the GPU with one untimed kernel launch before benchmarking
- Use `time.perf_counter()` for precise timing
- The crossover point where GPU beats CPU depends on the operation's arithmetic intensity

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-intro-gpu) for the solution and discussion.